# Breast Cancer Imaging ML Project
### Task 0: Declare Your Scope

**Body part / anatomical region:** Breast

**Imaging modality:** Mammography (X-ray)

**Problem type:** Semantic segmentation

**Target of interest:** Breast masses appearing in mammographic images. The model generates a pixel-level segmentation mask showing the location of the mass.

**Dataset:** CBIS-DDSM (Curated Breast Imaging Subset of DDSM)

**Dataset source:** https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset

**Date accessed:** 12/09/2026

**Real-world question:** Can an image segmentation model locate breast masses in mammograms by identifying the pixels associated with the abnormal region?

**Intended users:** This project is designed as an educational machine-learning prototype for demonstrating breast-image segmentation. It is not intended to replace a qualified healthcare professional or be used as a medical diagnostic system.

### Task 0a: Source Your Dataset

The project uses the CBIS-DDSM dataset, which is one of the breast mammography datasets listed in the assignment brief.

The dataset is publicly available through the listed Kaggle mirror. Kaggle requires a free account to access datasets for download.

**Dataset name:** CBIS-DDSM Breast Cancer Image Dataset

**Original dataset:** Curated Breast Imaging Subset of DDSM (CBIS-DDSM)

**Source:** Kaggle mirror of CBIS-DDSM

**Access date:** 12/09/2026

**Licence / usage terms:** The dataset's current licence and usage terms should be checked on the source page before submission. This project uses the dataset for educational coursework only and does not redistribute the underlying medical images.

The dataset is suitable for this project because it contains mammography images together with ROI information for breast masses, allowing the task to be formulated as semantic segmentation.

The dataset is separated into training and test records, and patient identifiers are available. Patient-level splitting is therefore used to reduce the risk of data leakage.

# A1. Problem Definition & Clinical Framing

This project investigates whether a U-Net semantic segmentation model can identify the location of breast masses in mammographic images.

The model does not directly make a diagnosis of breast cancer. Instead, it predicts a pixel-level mask indicating the region of the mammogram that corresponds to the annotated breast mass.

For this project:

**Input:** A grayscale mammogram.
**Target:** The annotated breast mass region.
**Output:** A binary segmentation mask.
**Positive pixels:** Pixels predicted to belong to the breast mass.
**Negative pixels:** Pixels predicted to belong to the background/non-mass region.

The project is a coursework prototype intended to demonstrate the development and evaluation of a medical-image segmentation model. It is not a medical diagnostic tool and should not be used to make clinical decisions.

The model assumes that the input is a mammographic image compatible with the type of images used during training. The input is treated as a single-channel grayscale image and resized to 256 × 256 pixels before being passed to the model.

The project therefore addresses the following question:

Can a semantic segmentation model learn to identify the pixels associated with annotated breast masses in mammographic images?

#### *Import the Python libraries we will need for data analysis, image processing, visualisation, and building our U-Net deep-learning model.*

In [3]:
# Data handling
import pandas as pd
import numpy as np

# File and directory handling
from pathlib import Path

# Image processing and visualisation
from PIL import Image
import matplotlib.pyplot as plt

# Deep learning
import tensorflow as tf


In [2]:
%pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## A2. Dataset Documentation

The CBIS-DDSM dataset contains mammography images together with metadata describing breast abnormalities and associated regions of interest (ROI). The dataset provides image information and ROI masks that can be used for breast-mass segmentation.

In [4]:
from pathlib import Path
import pandas as pd

# CBIS-DDSM dataset location
DATASET_DIR = Path("/home/Student/Documents/Machine_Learning_Journey/breast-cancer/CBIS-DDSM")
CSV_DIR = DATASET_DIR / "csv"

# Load the mass case description files
mass_train = pd.read_csv(
    CSV_DIR / "mass_case_description_train_set.csv"
)

mass_test = pd.read_csv(
    CSV_DIR / "mass_case_description_test_set.csv"
)

print("Training shape:", mass_train.shape)
print("Test shape:", mass_test.shape)

Training shape: (1318, 14)
Test shape: (378, 14)


In [5]:
print("Training columns:")
for i, column in enumerate(mass_train.columns, start=1):
    print(f"{i}. {column}")

print("\nFirst 3 training records:")
display(mass_train.head(3))

Training columns:
1. patient_id
2. breast_density
3. left or right breast
4. image view
5. abnormality id
6. abnormality type
7. mass shape
8. mass margins
9. assessment
10. pathology
11. subtlety
12. image file path
13. cropped image file path
14. ROI mask file path

First 3 training records:


,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00001,3,LEFT,CC,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...
1,P_00001,3,LEFT,MLO,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...
2,P_00004,3,LEFT,CC,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...


In [6]:
# Check the number of records and variables

print("Training records:", len(mass_train))
print("Test records:", len(mass_test))
print("Total records:", len(mass_train) + len(mass_test))
print("Number of variables:", mass_train.shape[1])

Training records: 1318
Test records: 378
Total records: 1696
Number of variables: 14


### Dataset size

The mass case description data is divided into training and test sets. The number of records in each set is examined below.

In [7]:
# Check the number of records and variables

mass_train.shape, mass_test.shape

((1318, 14), (378, 14))

### Missing values

The dataset is checked for missing values so that incomplete metadata can be identified before preprocessing.

In [8]:
# Check for missing values

print("Missing values in training data:")
display(mass_train.isnull().sum())

Missing values in training data:


patient_id                  0
breast_density              0
left or right breast        0
image view                  0
abnormality id              0
abnormality type            0
mass shape                  4
mass margins               43
assessment                  0
pathology                   0
subtlety                    0
image file path             0
cropped image file path     0
ROI mask file path          0
dtype: int64

### Segmentation annotations

The dataset provides an ROI mask path for each mass case. These masks provide the ground-truth regions that the U-Net will learn to segment.

In [9]:
# Examine the image and ROI mask paths

annotation_columns = [
    "image file path",
    "cropped image file path",
    "ROI mask file path"
]

display(mass_train[annotation_columns].head(3))

,image file path,cropped image file path,ROI mask file path
0,Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...
1,Mass-Training_P_00001_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...
2,Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...


### Pathology labels

The pathology labels describe the diagnosis associated with each mass case. Their distribution is examined to understand the composition of the dataset.

In [10]:
# Examine the distribution of pathology labels

print("Pathology distribution:")
display(mass_train["pathology"].value_counts(dropna=False))

Pathology distribution:


pathology
MALIGNANT                  637
BENIGN                     577
BENIGN_WITHOUT_CALLBACK    104
Name: count, dtype: int64

### Patient distribution

The number of unique patients is examined because multiple images can belong to the same patient. Patient-level separation will be used later to prevent data leakage between the training, validation, and test sets.

In [11]:
# Count unique patients in the training data

mass_train["patient_id"].nunique()

691

### Image information

The image dimensions and format are examined to understand the input data before preprocessing it for the segmentation model.

In [12]:
# Examine image and ROI mask paths

display(
    mass_train[["image file path", "ROI mask file path"]].head(3)
)

,image file path,ROI mask file path
0,Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...
1,Mass-Training_P_00001_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...
2,Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...


## A3. Data Preparation & Splitting

The mammogram images and their corresponding ROI masks will be prepared for U-Net segmentation. Images and masks must remain correctly paired throughout preprocessing. The data will also be split at the patient level to prevent information from the same patient appearing in multiple subsets.

In [13]:
from pathlib import Path
import pandas as pd

DATASET_DIR = Path(
    "/home/Student/Documents/Machine_Learning_Journey/breast-cancer/CBIS-DDSM"
)

CSV_DIR = DATASET_DIR / "csv"

mass_train = pd.read_csv(
    CSV_DIR / "mass_case_description_train_set.csv"
)

mass_test = pd.read_csv(
    CSV_DIR / "mass_case_description_test_set.csv"
)

print("Training shape:", mass_train.shape)
print("Test shape:", mass_test.shape)

Training shape: (1318, 14)
Test shape: (378, 14)


### Image and mask file paths

The dataset metadata contains paths to the mammogram images and ROI masks. A sample path is examined to determine how the files are organised before loading them.

In [14]:
# Examine one image path and one ROI mask path

mass_train[["image file path", "ROI mask file path"]].iloc[0]

image file path       Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590...
ROI mask file path    Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...
Name: 0, dtype: object

### Link mammograms with ROI masks

The image and ROI mask paths are linked using the directory structure provided in the dataset. Each mammogram is paired with its corresponding ground-truth mask for supervised segmentation.

In [15]:
from pathlib import Path
import pandas as pd

DATASET_DIR = Path(
    "/home/Student/Documents/Machine_Learning_Journey/breast-cancer/CBIS-DDSM"
)

CSV_DIR = DATASET_DIR / "csv"

dicom_info = pd.read_csv(
    CSV_DIR / "dicom_info.csv"
)

print("DICOM mapping shape:", dicom_info.shape)

DICOM mapping shape: (10237, 38)


In [16]:
# Link mammograms and ROI masks to their actual JPEG files

def find_jpeg(csv_path):
    uid = Path(str(csv_path).strip()).parent.name
    
    match = dicom_info[dicom_info["SeriesInstanceUID"] == uid]
    
    if len(match) == 0:
        return None
    
    image_path = str(match.iloc[0]["image_path"])
    image_path = Path(image_path)
    
    # Handle paths stored relative to the CBIS-DDSM folder
    if image_path.is_absolute():
        return str(image_path)
    
    if image_path.parts and image_path.parts[0] == "CBIS-DDSM":
        image_path = Path(*image_path.parts[1:])
    
    return str(DATASET_DIR / image_path)


mass_train["image_path"] = mass_train["image file path"].apply(find_jpeg)
mass_train["mask_path"] = mass_train["ROI mask file path"].apply(find_jpeg)

mass_test["image_path"] = mass_test["image file path"].apply(find_jpeg)
mass_test["mask_path"] = mass_test["ROI mask file path"].apply(find_jpeg)

print("Training images found:", mass_train["image_path"].notna().sum(), "/", len(mass_train))
print("Training masks found:", mass_train["mask_path"].notna().sum(), "/", len(mass_train))
print("Test images found:", mass_test["image_path"].notna().sum(), "/", len(mass_test))
print("Test masks found:", mass_test["mask_path"].notna().sum(), "/", len(mass_test))

Training images found: 1318 / 1318
Training masks found: 1318 / 1318
Test images found: 378 / 378
Test masks found: 378 / 378


In [17]:
# Verify that the linked files actually exist

print("Training image files existing:",
      sum(Path(p).exists() for p in mass_train["image_path"].dropna()))

print("Training mask files existing:",
      sum(Path(p).exists() for p in mass_train["mask_path"].dropna()))

print("Test image files existing:",
      sum(Path(p).exists() for p in mass_test["image_path"].dropna()))

print("Test mask files existing:",
      sum(Path(p).exists() for p in mass_test["mask_path"].dropna()))

print("\nExample image:", mass_train["image_path"].iloc[0])
print("Example mask:", mass_train["mask_path"].iloc[0])

Training image files existing: 1318
Training mask files existing: 1318
Test image files existing: 378
Test mask files existing: 378

Example image: /home/Student/Documents/Machine_Learning_Journey/breast-cancer/CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.342386194811267636608694132590482924515/1-211.jpg
Example mask: /home/Student/Documents/Machine_Learning_Journey/breast-cancer/CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.296736403313792599626368780122205399650/1-250.jpg


### Patient-level train and validation split

The training data is divided into training and validation subsets at the patient level rather than randomly splitting individual images. This prevents images from the same patient from appearing in both subsets and reduces the risk of data leakage. The original dataset test set is kept separate for final evaluation.

In [18]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        mass_train,
        groups=mass_train["patient_id"]
    )
)

train_data = mass_train.iloc[train_idx].copy()
val_data = mass_train.iloc[val_idx].copy()
test_data = mass_test.copy()

train_patients = set(train_data["patient_id"])
val_patients = set(val_data["patient_id"])
test_patients = set(test_data["patient_id"])

# Confirm there is no patient overlap
assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

split_summary = pd.DataFrame({
    "Subset": ["Training", "Validation", "Test"],
    "Images": [
        len(train_data),
        len(val_data),
        len(test_data)
    ],
    "Patients": [
        train_data["patient_id"].nunique(),
        val_data["patient_id"].nunique(),
        test_data["patient_id"].nunique()
    ]
})

display(split_summary)

,Subset,Images,Patients
0,Training,1061,552
1,Validation,257,139
2,Test,378,201


### Image preprocessing

The mammograms and corresponding ROI masks are resized to 256 × 256 pixels. Mammogram pixel values are normalised to the range 0–1. ROI masks use nearest-neighbour interpolation so that their binary segmentation labels remain intact.

In [19]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

### TensorFlow data pipeline

A TensorFlow input pipeline loads mammograms and their corresponding ROI masks directly from disk. Images and masks remain paired throughout preprocessing, while batching and prefetching improve the training workflow.

In [20]:
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE


def load_image_mask(image_path, mask_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=1)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_jpeg(mask, channels=1)
    mask = tf.image.resize(
        mask,
        IMAGE_SIZE,
        method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
    )
    mask = tf.cast(mask > 127, tf.float32)

    return image, mask


def create_dataset(dataframe, training=False):
    image_paths = dataframe["image_path"].astype(str).tolist()
    mask_paths = dataframe["mask_path"].astype(str).tolist()

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, mask_paths)
    )

    dataset = dataset.map(
        load_image_mask,
        num_parallel_calls=AUTOTUNE
    )

    if training:
        dataset = dataset.shuffle(
            len(dataframe),
            seed=42
        )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

### Dataset construction

Separate TensorFlow datasets are created for training, validation, and testing. Training data is shuffled, while validation and test data are kept deterministic for evaluation.

In [21]:
# Create TensorFlow datasets

train_ds = create_dataset(train_data, training=True)
val_ds = create_dataset(val_data, training=False)
test_ds = create_dataset(test_data, training=False)

print("Training dataset created.")
print("Validation dataset created.")
print("Test dataset created.")

Training dataset created.
Validation dataset created.
Test dataset created.


2026-09-16 18:13:46.046989: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


### Preprocessing verification

A sample batch is inspected to confirm that the images and masks have the expected dimensions and value ranges before model training.

In [22]:
sample_images, sample_masks = next(iter(train_ds))

pd.DataFrame({
    "Property": [
        "Image shape",
        "Mask shape",
        "Image minimum",
        "Image maximum",
        "Mask minimum",
        "Mask maximum"
    ],
    "Value": [
        sample_images.shape,
        sample_masks.shape,
        float(tf.reduce_min(sample_images)),
        float(tf.reduce_max(sample_images)),
        float(tf.reduce_min(sample_masks)),
        float(tf.reduce_max(sample_masks))
    ]
})

,Property,Value
0,Image shape,"(8, 256, 256, 1)"
1,Mask shape,"(8, 256, 256, 1)"
2,Image minimum,0.0
3,Image maximum,1.0
4,Mask minimum,0.0
5,Mask maximum,1.0


# A4. Model Architecture & Justification

A U-Net convolutional neural network is used for supervised semantic segmentation. The encoder learns increasingly complex image features while the decoder reconstructs a pixel-level segmentation map. Skip connections transfer spatial information from the encoder to the decoder, helping the model localise regions of interest.

U-Net was selected because it is a well-established architecture for medical image segmentation, provides a strong baseline for this project, and produces a pixel-level output that can be compared directly with the ground-truth ROI masks.

In [23]:
from tensorflow.keras import layers, Model


def conv_block(x, filters):
    x = layers.Conv2D(
        filters, 3, padding="same", activation="relu"
    )(x)
    x = layers.Conv2D(
        filters, 3, padding="same", activation="relu"
    )(x)
    return x


def build_unet(input_shape=(256, 256, 1)):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256)
    p4 = layers.MaxPooling2D()(c4)

    # Bottleneck
    c5 = conv_block(p4, 512)

    # Decoder
    u6 = layers.Conv2DTranspose(
        256, 2, strides=2, padding="same"
    )(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = conv_block(u6, 256)

    u7 = layers.Conv2DTranspose(
        128, 2, strides=2, padding="same"
    )(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = conv_block(u7, 128)

    u8 = layers.Conv2DTranspose(
        64, 2, strides=2, padding="same"
    )(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = conv_block(u8, 64)

    u9 = layers.Conv2DTranspose(
        32, 2, strides=2, padding="same"
    )(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = conv_block(u9, 32)

    outputs = layers.Conv2D(
        1, 1, activation="sigmoid"
    )(c9)

    return Model(inputs, outputs)


model = build_unet()

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │        320 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │      9,248 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │     36,928 │ conv2d_2[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 64,    │    147,584 │ conv2d_4[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 32, 32,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 32, 32,    │    295,168 │ max_pooling2d_2[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 32, 32,    │    590,080 │ conv2d_6[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 16, 16,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 16, 16,    │  1,180,160 │ max_pooling2d_3[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 16, 16,    │  2,359,808 │ conv2d_8[0][0]    │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 32, 32,    │    524,544 │ conv2d_9[0][0]    │
│ (Conv2DTranspose)   │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32, 32,    │          0 │ conv2d_transpose

 Total params: 7,759,521 (29.60 MB)

 Trainable params: 7,759,521 (29.60 MB)

 Non-trainable params: 0 (0.00 B)

# A5. Training

The model is trained using a combined Binary Cross-Entropy and Dice loss. Binary Cross-Entropy measures pixel-level classification error, while Dice loss encourages overlap between the predicted segmentation and the ground-truth ROI.

The Adam optimiser is used with a learning rate of 0.0001. Early stopping is used to reduce overfitting by stopping training when validation Dice performance stops improving.

In [24]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [25]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)

    return (
        (2.0 * intersection + smooth)
        /
        (
            tf.reduce_sum(y_true)
            + tf.reduce_sum(y_pred)
            + smooth
        )
    )


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)


def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(
        y_true, y_pred
    )

    return tf.reduce_mean(bce) + dice_loss(
        y_true, y_pred
    )


model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=combined_loss,
    metrics=[dice_coefficient]
)

print("Model compiled successfully.")

Model compiled successfully.


### Model training

The U-Net is trained using the patient-level training set and monitored using the validation set. Early stopping is used to reduce the risk of overfitting.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_dice_coefficient",
        mode="max",
        patience=5,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=6,
    callbacks=callbacks
)

Epoch 1/6
 18/133 ━━━━━━━━━━━━━━━━━━━━ 5:25 3s/step - dice_coefficient: 0.3687 - loss: 1.3244

### Training and validation curves

Training and validation Dice performance are visualised to assess learning progress and identify possible overfitting. A large difference between training and validation performance may indicate that the model is fitting the training data more strongly than unseen data.

In [ ]:
history_df = pd.DataFrame(history.history)

print(history_df)

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["dice_coefficient"], label="Training Dice")
plt.plot(history_df["val_dice_coefficient"], label="Validation Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice Coefficient")
plt.title("Training and Validation Dice")
plt.legend()
plt.show()

# A6. Evaluation

The final model is evaluated on the original CBIS-DDSM test set, which was not used during model training or validation. Dice coefficient and Intersection over Union (IoU) are used because segmentation quality depends on the spatial overlap between the predicted ROI and the ground-truth mask.

In [ ]:
test_results = model.evaluate(
    test_ds,
    return_dict=True,
    verbose=1
)

display(pd.DataFrame({
    "Metric": list(test_results.keys()),
    "Value": list(test_results.values())
}))

### Intersection over Union

Intersection over Union (IoU) measures the overlap between the predicted segmentation region and the ground-truth ROI. Higher IoU indicates greater spatial agreement between the prediction and the annotation.

In [ ]:
def calculate_iou(y_true, y_pred, threshold=0.5, smooth=1e-6):
    y_pred = tf.cast(y_pred > threshold, tf.float32)
    y_true = tf.cast(y_true > 0.5, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred)

    union = (
        tf.reduce_sum(y_true)
        + tf.reduce_sum(y_pred)
        - intersection
    )

    return (
        intersection + smooth
    ) / (
        union + smooth
    )

### Test-set IoU evaluation

IoU is calculated for each test image and then averaged across the test set. This provides an additional segmentation metric alongside Dice coefficient.

In [ ]:
ious = []

for images, masks in test_ds:
    predictions = model.predict(images, verbose=0)

    for true_mask, pred_mask in zip(masks, predictions):
        iou = calculate_iou(true_mask, pred_mask)
        ious.append(float(iou))

print("Mean IoU:", np.mean(ious))

### Qualitative segmentation results

A correct and an incorrect test-set prediction are visualised alongside their ground-truth masks. Showing both examples provides a balanced assessment of model performance rather than presenting only successful predictions.

In [ ]:
results = []

for images, masks in test_ds:
    predictions = model.predict(images, verbose=0)

    for i in range(len(images)):
        iou = calculate_iou(
            masks[i],
            predictions[i]
        )

        results.append({
            "image": images[i].numpy(),
            "mask": masks[i].numpy(),
            "prediction": predictions[i].numpy(),
            "iou": float(iou)
        })

best_example = max(results, key=lambda x: x["iou"])
worst_example = min(results, key=lambda x: x["iou"])

pd.DataFrame({
    "Example": ["Higher-overlap example", "Lower-overlap example"],
    "IoU": [
        best_example["iou"],
        worst_example["iou"]
    ]
})

In [ ]:
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.imshow(
    best_example["image"][:, :, 0],
    cmap="gray"
)
plt.title("Better Example: Mammogram")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(
    best_example["mask"][:, :, 0],
    cmap="gray"
)
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(
    best_example["prediction"][:, :, 0] > 0.5,
    cmap="gray"
)
plt.title("Prediction")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(
    worst_example["image"][:, :, 0],
    cmap="gray"
)
plt.title("Lower-IoU Example: Mammogram")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(
    worst_example["mask"][:, :, 0],
    cmap="gray"
)
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(
    worst_example["prediction"][:, :, 0] > 0.5,
    cmap="gray"
)
plt.title("Prediction")
plt.axis("off")

plt.tight_layout()
plt.show()

# A7. Model Export

The trained U-Net model is exported in Keras format. The saved model will be used by the Streamlit application.

The Streamlit application must use the same image size, grayscale input format, and 0–1 pixel normalisation used during model training.

In [ ]:
model.save("breast_mass_unet.keras")

# A8. Limitations, Ethics & Next Steps

### Limitations

This project is an educational prototype and is not a medical diagnostic system. The model is trained on the CBIS-DDSM dataset and may not generalise to mammograms from different populations, hospitals, scanners, acquisition protocols, or preprocessing pipelines.

The ROI annotations represent the available dataset annotations and may not perfectly represent every clinically relevant region. Resizing the original mammograms to 256 × 256 pixels also reduces spatial detail.

Model performance may vary between images, as demonstrated by the difference between higher- and lower-IoU examples.

### Ethical considerations

The model should not be used to make clinical decisions. Medical imaging models can produce incorrect predictions, so predictions should not replace assessment by qualified healthcare professionals.

Patient privacy must also be protected when handling medical imaging data.

### Next steps

Future improvements could include stronger data augmentation, hyperparameter tuning, transfer learning, improved segmentation architectures such as U-Net++, and evaluation on an independent external dataset.

The Streamlit application will be presented clearly as an educational demonstration rather than a diagnostic tool.

In [ ]:
import os
from pathlib import Path
import tensorflow as tf

model_path = Path("breast_mass_unet.keras")

model.save(model_path)

print("Model saved:", model_path.exists())
print("Model path:", model_path.resolve())
print("TensorFlow version:", tf.__version__)
print("Keras version:", tf.keras.__version__)

In [ ]:
print("model" in globals())

In [ ]:
import os
print(os.path.exists("breast_mass_unet.keras"))

In [ ]:
import os
print(os.path.abspath("breast_mass_unet.keras"))

In [1]:
print("Notebook is working")
print(2 + 2)

Notebook is working
4
